# NLUM Land Use Cleaning (Colab)

This Colab notebook downloads a public land use raster (NLUM), clips a small subset,
and cleans it using `multiclean.clean_array`.

In [ ]:
# Install dependencies (Colab)
%pip -q install --upgrade pip
%pip -q install multiclean rasterio matplotlib numpy scipy opencv-python-headless

In [ ]:
import os
import zipfile
from pathlib import Path
import requests
import numpy as np
import matplotlib.pyplot as plt
import rasterio as rio
from multiclean import clean_array

DATA_URL = ("https://www.agriculture.gov.au/sites/default/files/documents/"            "NLUM_v7_250_ALUMV8_2010_11_alb_package_20241128.zip")
WORKDIR = Path.cwd() / 'nlum_data'
WORKDIR.mkdir(exist_ok=True)
ZIP_PATH = WORKDIR / 'nlum.zip'
EXTRACT_DIR = WORKDIR / 'extracted'
EXTRACT_DIR.mkdir(exist_ok=True)

print('Working directory:', WORKDIR)

In [ ]:
# Download the NLUM zip
if not ZIP_PATH.exists():
    print('Downloading...', DATA_URL)
    with requests.get(DATA_URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(ZIP_PATH, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk: f.write(chunk)
    print('Saved to', ZIP_PATH)
else:
    print('Zip already exists at', ZIP_PATH)

# Extract
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_DIR)
print('Extracted to', EXTRACT_DIR)

In [ ]:
# Locate the TIFF: NLUM_v7_250_ALUMV8_2010_11_alb.tif
candidates = list(EXTRACT_DIR.rglob('*.tif'))
if not candidates:
    raise FileNotFoundError('No .tif files found in extracted package')
# Prefer the expected filename if present
preferred = [p for p in candidates if 'NLUM_v7_250_ALUMV8_2010_11_alb.tif' in p.name]
tif_path = preferred[0] if preferred else candidates[0]
print('Using TIFF:', tif_path)

with rio.open(tif_path) as src:
    full = src.read(1)  # 2D array of class labels

print('Raster shape:', full.shape, 'dtype:', full.dtype)
print('Sample classes:', np.unique(full)[:15])

In [ ]:
# Create a 500x500 pixel subset near the center (adjusts for edges)
h, w = full.shape
win = 500
r0 = max(0, h // 2 - win // 2)
c0 = max(0, w // 2 - win // 2)
r1 = min(h, r0 + win)
c1 = min(w, c0 + win)
subset = full[r0:r1, c0:c1].astype(np.int32, copy=False)
print('Subset shape:', subset.shape)

# Visualize the raw subset
plt.figure(figsize=(6,6))
plt.imshow(subset, cmap='tab20', interpolation='nearest')
plt.title('Raw NLUM subset')
plt.axis('off')
plt.show()

print('Unique classes in subset:', np.unique(subset))

In [ ]:
# Clean the subset
cleaned = clean_array(
    array=subset,
    smooth_edge_size=2,
    min_island_size=100,
    connectivity=8,
)

# Compare before/after
fig, axes = plt.subplots(1, 2, figsize=(12,6))
axes[0].imshow(subset, cmap='tab20', interpolation='nearest')
axes[0].set_title('Before')
axes[0].axis('off')
axes[1].imshow(cleaned, cmap='tab20', interpolation='nearest')
axes[1].set_title('After (MultiClean)')
axes[1].axis('off')
plt.show()

print('Unique classes after:', np.unique(cleaned))

## Notes
- Adjust `smooth_edge_size` and `min_island_size` to match your data.
- The subset is taken by pixel indices; for geospatial windows, use `rasterio.windows`.
- `multiclean` keeps integer classes intact and fills small gaps via nearest-class assignment.